# Physics-aware vs statistical anomaly detection in solar PV — advanced study
Adds to the base project: **multi-seed results (mean ± std)**, **Isolation Forest and GRU baselines**, **hybrid physics + deep-learning detector**, **adversarial training with adaptive attacks**, a **projection-only control**, and **precision / recall / F1 / ROC-AUC / confusion matrices**.

First run with `QUICK = True` (smoke test, a few minutes). Then set `QUICK = False` and run again for the final numbers (about 3 seeds; can take 30–60 min).

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
import copy, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# >>> change this if your CSVs are somewhere else <<<
DATA_DIR = r"C:\\Users\\Shreya\\Downloads"
TRAIN_CSV = os.path.join(DATA_DIR, "pv_train_merged.csv")
TEST_CSV  = os.path.join(DATA_DIR, "pv_test_merged.csv")

# >>> QUICK=True is a fast smoke test (1 seed). Set QUICK=False for the final multi-seed run. <<<
QUICK = True
if QUICK: SEEDS, EPOCHS, ATT_N, ADV_EPOCHS = [0], 12, 200, 5
else:     SEEDS, EPOCHS, ATT_N, ADV_EPOCHS = [0, 1, 2], 30, 400, 10
EPSS, EPS_MAIN = [0.02, 0.05, 0.1, 0.2], 0.1
ATTACKS = [("FGSM", 1, False), ("PGD", 20, False), ("PGD+physics", 20, True)]
F = ["G", "T", "V", "I", "P"]
L, S = 24, 6
rng = np.random.default_rng(0)
os.makedirs("figures", exist_ok=True)

## PV environment, faults, physics rules

In [ ]:
# ===== PV environment: single-diode model, P = V x I =====
ISC, VOC, NS, NID = 8.2, 37.0, 60, 1.3
def a_of(T):  return NID * NS * 0.02585 * (T + 273.15) / 298.15
def i0_of(T): return ISC / (np.exp(VOC / a_of(25.0)) - 1) * np.exp(0.065 * (T - 25))
def iph(G, T): return ISC * (G / 1000) * (1 + 0.0005 * (T - 25))
def i_curve(V, G, T):
    return np.maximum(iph(G, T) - i0_of(T) * (np.exp(np.minimum(V / a_of(T), 50)) - 1), 0)
def mpp(G, T):
    Vg = np.linspace(0, VOC, 300)[None, :]
    I = i_curve(Vg, G[:, None], T[:, None]); P = Vg * I
    k = P.argmax(1)
    return Vg[0][k], I[np.arange(len(k)), k]

def simulate_from_weather(path):
    d = pd.read_csv(path, usecols=["instantaneous_global_radiation", "clearsky_Tamb", "radiation_mean", "radiation_std",
                                   "tprt_mean", "tprt_std", "label", "site_num", "date", "source_file"])
    d = d[d.label == 0]
    G = np.clip(d.instantaneous_global_radiation * d.radiation_std + d.radiation_mean, 0, None).to_numpy()
    T = (d.clearsky_Tamb * d.tprt_std + d.tprt_mean).to_numpy() + 0.03 * G
    V, I = np.zeros(len(G)), np.zeros(len(G))
    for s in range(0, len(G), 4000):
        V[s:s+4000], I[s:s+4000] = mpp(G[s:s+4000], T[s:s+4000])
    V *= 1 + .005 * rng.standard_normal(len(G)); I *= 1 + .005 * rng.standard_normal(len(G))
    P = V * I * (1 + .003 * rng.standard_normal(len(G)))
    X = np.stack([G, T, V, I, P], 1).astype(np.float32)
    keys = d.groupby(["source_file", "site_num", "date"], sort=False).ngroup().to_numpy()
    return X, keys, d.site_num.to_numpy()

FAULTS = {1: "partial_shading", 2: "sensor_drift", 3: "inverter_fault"}
def inject(X):
    X = X.copy(); ft = np.zeros(len(X), int)
    if len(X) < 40: return X, ft
    for _ in range(len(X) // 40):
        s = rng.integers(0, len(X) - 20); e = s + rng.integers(8, 20); kind = int(rng.integers(1, 4))
        if kind == 1:
            X[s:e, 3] *= rng.uniform(.4, .7); X[s:e, 2] *= rng.uniform(.9, .97); X[s:e, 4] = X[s:e, 2] * X[s:e, 3]
        elif kind == 2:
            X[s:e, 2] *= 1 + np.linspace(0, rng.uniform(.1, .25), e - s)
        else:
            X[s:e, 3] *= rng.uniform(0, .2); X[s:e, 4] = X[s:e, 2] * X[s:e, 3]
        ft[s:e] = kind
    return X, ft

def build(X, keys, site, faults):
    W, T, SI = [], [], []
    cut = np.flatnonzero(np.diff(keys)) + 1
    for xs, st in zip(np.split(X, cut), np.split(site, cut)):
        ft = np.zeros(len(xs), int)
        if faults: xs, ft = inject(xs)
        for i in range(0, len(xs) - L + 1, S):
            w = xs[i:i+L]; f = ft[i:i+L]; nf = (f > 0).sum()
            if w[:, 0].mean() < 150 or 0 < nf < 4: continue
            W.append(w); T.append(f.max()); SI.append(st[0])
    return np.stack(W), np.array(T), np.array(SI)

# scaling + physics rules (LO/HI are set per seed in run_seed)
LO = HI = None
to_norm = lambda x: np.clip((x - LO) / (HI - LO), 0, 1).astype(np.float32)
to_raw  = lambda xn: xn * (HI - LO) + LO
TOL = 0.15
def project(xn):
    """Project onto single-diode physics: I near the I-V curve for (V,G,T), then P = V*I."""
    r = to_raw(xn)
    G, T, V = np.maximum(r[..., 0], 0), r[..., 1], np.clip(r[..., 2], 0, VOC)
    Ic = i_curve(V, G, T)
    I = np.clip(r[..., 3], Ic * (1 - TOL), Ic * (1 + TOL))
    return to_norm(np.stack([G, T, V, I, V * I], -1))
def physical_ok(xn, slack=0.05):
    r = to_raw(xn)
    G, T, V, I, P = [r[..., k] for k in range(5)]
    Ic = i_curve(V, G, T)
    return ((np.abs(P - V * I) / (np.abs(P) + 1) < 0.02) & (np.abs(I - Ic) / (Ic + .1) < TOL + slack)).all(1)

## Models, attacks and defences

In [ ]:
# ===== models, attacks, defences =====
class AE(nn.Module):
    def __init__(s, f=5, h=32, kind="lstm"):
        super().__init__()
        R = nn.LSTM if kind == "lstm" else nn.GRU
        s.kind = kind; s.enc = R(f, h, batch_first=True); s.dec = R(h, h, batch_first=True); s.out = nn.Linear(h, f)
    def forward(s, x):
        _, hh = s.enc(x)
        h = hh[0] if s.kind == "lstm" else hh
        z = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        return s.out(s.dec(z)[0])

def err(model, x):
    with torch.no_grad():
        x = torch.as_tensor(x); return ((model(x) - x) ** 2).mean((1, 2)).numpy()

def train_ae(model, Xn, epochs):
    opt = torch.optim.Adam(model.parameters(), 1e-3); X = torch.as_tensor(Xn)
    for _ in range(epochs):
        for b in torch.randperm(len(X)).split(64):
            loss = ((model(X[b]) - X[b]) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); return model

def attack(model, xn, eps, steps, alpha, physics):
    x0 = torch.as_tensor(xn); x = x0.clone()
    for _ in range(steps):
        x.requires_grad_(True)
        loss = ((model(x) - x) ** 2).mean((1, 2)).sum()
        g, = torch.autograd.grad(loss, x)
        x = x.detach() - alpha * g.sign()
        x = torch.min(torch.max(x, x0 - eps), x0 + eps).clamp(0, 1)
        if physics: x = torch.as_tensor(project(x.numpy()))
    return x.detach().numpy()

def adv_finetune(model, Xn_norm, Xn_fault, epochs, lam=3.0):
    """Adversarial training for an anomaly detector: keep reconstructing normal data well, and force a
    HIGH error on faulty windows, including windows already attacked (FGSM-like/PGD and physics-PGD)."""
    opt = torch.optim.Adam(model.parameters(), 5e-4); X = torch.as_tensor(Xn_norm)
    margin = float(4 * np.percentile(err(model, Xn_norm), 99))
    for _ in range(epochs):
        idx = rng.choice(len(Xn_fault), size=min(600, len(Xn_fault)), replace=False)
        Fc = Xn_fault[idx]; h = len(Fc) // 2
        pool = np.concatenate([Fc, attack(model, Fc[:h], 0.1, 10, 0.02, False),
                               attack(model, Fc[h:], 0.1, 10, 0.02, True)]).astype(np.float32)
        pool = torch.as_tensor(pool)
        for b in torch.randperm(len(X)).split(64):
            xf = pool[torch.randint(len(pool), (len(b),))]
            e_f = ((model(xf) - xf) ** 2).mean((1, 2))
            loss = ((model(X[b]) - X[b]) ** 2).mean() + lam * torch.relu(margin - e_f).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); return model

def feats(xn):   # purely statistical window features for Isolation Forest (no physics)
    d = np.abs(np.diff(xn, axis=1))
    return np.concatenate([xn.mean(1), xn.std(1), xn.min(1), xn.max(1), d.mean(1)], 1)

def f1_from(rate, fpr, npos, nneg):
    tp = rate * npos; fn = npos - tp; fp = fpr * nneg
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

## Run experiments (trains all detectors and attacks them, per seed)

In [ ]:
clean_rows, att_rows, valid_rows, proj_rows, S0 = [], [], [], [], {}

def run_seed(seed):
    global rng, LO, HI
    rng = np.random.default_rng(seed); torch.manual_seed(seed); t0 = time.time()
    # --- data (new noise, faults and splits for every seed) ---
    Xs, ks, ss = simulate_from_weather(TRAIN_CSV)
    Xn_all, _, sn = build(Xs, ks, ss, faults=False)
    Xf_all, tf, sf = build(Xs, ks, ss, faults=True)
    Xs, ks, ss = simulate_from_weather(TEST_CSV)
    Xte, tte, _ = build(Xs, ks, ss, faults=True); yte = (tte > 0).astype(int)
    val_sites = np.unique(sn)[-3:]; isval = np.isin(sn, val_sites)
    Xfit, Xval = Xn_all[~isval], Xn_all[isval]
    Xfault_train = Xf_all[(tf > 0) & ~np.isin(sf, val_sites)]
    rows = Xfit.reshape(-1, 5); span = rows.max(0) - rows.min(0)
    LO, HI = rows.min(0) - .05 * span, rows.max(0) + .05 * span
    Xfit_n, Xval_n, Xft_n, Xt = to_norm(Xfit), to_norm(Xval), to_norm(Xfault_train), to_norm(Xte)
    N = Xt[yte == 0]; nneg = len(N)
    print(f"[seed {seed}] fit {len(Xfit)} | val {len(Xval)} | fault-train {len(Xft_n)} | test {len(Xt)} (faulty {yte.sum()})")

    # --- detectors ---
    base = train_ae(AE(kind="lstm"), Xfit_n, EPOCHS)
    gru  = train_ae(AE(kind="gru"),  Xfit_n, EPOCHS)
    iso  = IsolationForest(n_estimators=200, random_state=seed).fit(feats(Xfit_n))
    score_if = lambda x: -iso.decision_function(feats(x))
    thr_b, thr_g = np.percentile(err(base, Xval_n), 99), np.percentile(err(gru, Xval_n), 99)
    thr_i = np.percentile(score_if(Xval_n), 99)
    D0 = {"LSTM-AE": lambda x: err(base, x) > thr_b, "GRU-AE": lambda x: err(gru, x) > thr_g,
          "IsoForest": lambda x: score_if(x) > thr_i, "Physics": lambda x: ~physical_ok(x)}
    D0["Hybrid"] = lambda x: D0["LSTM-AE"](x) | D0["Physics"](x)
    scores0 = {"LSTM-AE": lambda x: err(base, x), "GRU-AE": lambda x: err(gru, x), "IsoForest": score_if}

    # --- adversarially trained detector ---
    adv = adv_finetune(copy.deepcopy(base), Xfit_n, Xft_n, ADV_EPOCHS)
    thr_a = np.percentile(err(adv, Xval_n), 99)
    D1 = {"Adv-AE": lambda x: err(adv, x) > thr_a}
    D1["Hybrid-adv"] = lambda x: D1["Adv-AE"](x) | ~physical_ok(x)
    scores1 = {"Adv-AE": lambda x: err(adv, x)}

    # --- clean metrics ---
    flags = {}
    for D, sc in [(D0, scores0), (D1, scores1)]:
        for name, fn in D.items():
            f = fn(Xt).astype(int); flags[name] = f
            p, r, f1, _ = precision_recall_fscore_support(yte, f, average="binary", zero_division=0)
            auc = roc_auc_score(yte, sc[name](Xt)) if name in sc else np.nan
            clean_rows.append(dict(seed=seed, detector=name, precision=p, recall=r, f1=f1, fpr=f[yte == 0].mean(), auc=auc, nneg=nneg))
    if seed == SEEDS[0]: S0.update(yte=yte, flags=flags)

    # --- attacks ---
    A = Xt[rng.choice(np.where(yte == 1)[0], size=min(ATT_N, int(yte.sum())), replace=False)]
    pr = project(A)
    proj_rows.append(dict(seed=seed, evasion=1 - D0["LSTM-AE"](pr).mean(), pert=np.abs(pr - A).max((1, 2)).mean()))
    for target, model, D in [("base", base, D0), ("adv", adv, D1)]:       # adv = ADAPTIVE attack on the hardened model
        for eps in EPSS:
            for name, steps, phys in ATTACKS:
                xa = attack(model, A, eps, steps, eps if steps == 1 else eps / 5, phys)
                valid_rows.append(dict(seed=seed, target=target, attack=name, eps=eps,
                                       valid=physical_ok(xa).mean(), pert=np.abs(xa - A).max((1, 2)).mean()))
                for dn, fn in D.items():
                    att_rows.append(dict(seed=seed, target=target, attack=name, eps=eps, detector=dn, rate=fn(xa).mean(), npos=len(A)))
    print(f"[seed {seed}] done in {(time.time()-t0)/60:.1f} min")

for sd in SEEDS: run_seed(sd)
clean, att, valid, proj = map(pd.DataFrame, (clean_rows, att_rows, valid_rows, proj_rows))

## Results tables

In [ ]:
def ms(df, index, col, val):
    g = df.groupby(index + [col])[val].agg(["mean", "std"]).fillna(0)
    return (g["mean"].map("{:.2f}".format) + " ± " + g["std"].map("{:.2f}".format)).unstack(col)

print("===== CLEAN TEST (no attack): mean ± std over", len(SEEDS), "seed(s) =====")
tabs = {m: clean.groupby("detector")[m].agg(["mean", "std"]) for m in ["precision", "recall", "f1", "fpr", "auc"]}
fmt = lambda v: "n/a" if pd.isna(v) else f"{v:.2f}"
clean_tab = pd.DataFrame({m: t["mean"].map(fmt) + (" ± " + t["std"].fillna(0).map(fmt)).where(t["mean"].notna(), "") for m, t in tabs.items()})
print(clean_tab.to_string())

att = att.merge(clean[["seed", "detector", "fpr", "nneg"]], on=["seed", "detector"])
att["f1"] = [f1_from(r, f, n, m) for r, f, n, m in zip(att.rate, att.fpr, att.npos, att.nneg)]
m_ = att[att.eps == EPS_MAIN]
print(f"\n===== DETECTION RATE UNDER ATTACK (eps={EPS_MAIN}); higher = defender wins =====")
print(ms(m_, ["target", "attack"], "detector", "rate").to_string())
print(f"\n===== F1 UNDER ATTACK (eps={EPS_MAIN}) =====")
print(ms(m_, ["target", "attack"], "detector", "f1").to_string())
print("\n===== PHYSICAL VALIDITY and attack size =====")
print(ms(valid, ["target", "attack"], "eps", "valid").to_string()); print(); print(ms(valid, ["target", "attack"], "eps", "pert").to_string())
print("\n===== PROJECTION ONLY (no gradient attack): how much of the physics attack is just 'repairing' the fault? =====")
print(proj.mean().round(2).to_string())

clean_tab.to_csv("results_clean.csv"); att.to_csv("results_attacks.csv", index=False); valid.to_csv("results_validity.csv", index=False)

## Plots

In [ ]:
def plot_target(target, dets, title, fname):
    fig, axs = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
    for a, atk in zip(axs, [x[0] for x in ATTACKS]):
        for d in dets:
            s = att[(att.target == target) & (att.attack == atk) & (att.detector == d)].groupby("eps").rate.agg(["mean", "std"]).fillna(0)
            a.errorbar(s.index, s["mean"], yerr=s["std"], marker="o", capsize=3, label=d)
        a.set_title(atk); a.set_xlabel("epsilon"); a.set_ylim(-.05, 1.05)
    axs[0].set_ylabel("detection rate"); axs[0].legend(fontsize=7); fig.suptitle(title); plt.tight_layout()
    plt.savefig(fname, dpi=150); plt.show()

plot_target("base", ["LSTM-AE", "GRU-AE", "IsoForest", "Physics", "Hybrid"], "Attacks crafted against the LSTM-AE (others = transfer)", "figures/detection_base.png")
plot_target("adv", ["Adv-AE", "Hybrid-adv"], "ADAPTIVE attacks against the adversarially trained detector", "figures/detection_adv.png")

# confusion matrices (first seed)
yte0 = S0["yte"]
fig, axs = plt.subplots(1, 4, figsize=(16, 3.6))
for a, n in zip(axs, ["LSTM-AE", "Physics", "Hybrid", "Adv-AE"]):
    ConfusionMatrixDisplay(confusion_matrix(yte0, S0["flags"][n]), display_labels=["normal", "fault"]).plot(ax=a, colorbar=False); a.set_title(n)
plt.tight_layout(); plt.savefig("figures/confusion_matrices.png", dpi=150); plt.show()